# Binary Anomaly Detection Model

This notebook trains supervised machine learning models to classify
OpenStack log events as Normal or Anomalous.

Responsibilities:

- Load engineered features
- Perform model-specific preprocessing
- Handle remaining missing values
- Train multiple classification models
- Track experiments using MLflow
- Compare model performance
- Save the best model

Algorithms:

- Logistic Regression
- Decision Tree
- Random Forest

Target Column:

is_anomaly

In [0]:
%run ./06_ML_Utilities

In [0]:
## Read The Feature Dataset

ml_df = spark.table("`log-analytics`.gold.feature_engineering_dataset")

print("="*60)
print("ML FEATURE DATASET")
print("="*60)

print(f"Rows   : {ml_df.count():,}")
print(f"Columns: {len(ml_df.columns)}")

display(ml_df.limit(5))

In [0]:
## Dataset Validation
print("="*60)
print("TARGET DISTRIBUTION")
print("="*60)

display(
    ml_df.groupBy("is_anomaly").count()
)

In [0]:
ml_df.printSchema()

#### Model Specific Feature Selection
we will drop columns which are are no use for anomaly detection, the features we will drop are:

- request_id
- user_id
- project_id
- instance_id
- client_ip
- http_path
- message
- kafka_offset
- kafka_timestamp
- ingestion_timestamp
- kafka_partition
- kafka_topic
- process_id

we will keep remaining columns as they are important

In [0]:
## Model Specific Feature Selection
binary_df = ml_df.drop(
    "request_id",
    "instance_id",
    "user_id",
    "project_id",
    "client_ip",
    "http_path",
    "message",
    "kafka_offset",
    "kafka_timestamp",
    "ingestion_timestamp",
    "kafka_partition",
    "kafka_topic",
    "process_id"
)

display(binary_df.limit(10))

#### Remaining Null Handling

In [0]:
from pyspark.sql import functions as F

null_summary = binary_df.select([
    F.count(
        F.when(F.col(c).isNull(),c)
    ).alias(c)

    for c in binary_df.columns
])

display(null_summary)

In [0]:
## Define Numerical And Categorical Features
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

## Categorical Features
categorical_cols = [
    "service",
    "log_file",
    "log_level",
    "http_method",
    "response_category"
]

## Numerical Features
numerical_cols = [
    "status_code",
    "response_time",
    "event_hour",
    "event_day",
    "event_month",
    "event_weekday",
    "is_weekend",
    "instance_exists"
]

target = "is_anomaly"

In [0]:
## Train test Split
train_df, test_df = binary_df.randomSplit(
    [0.8, 0.2],
    seed = 42
)

print("="*60)
print("TRAIN TEST SPLIT")
print("="*60)

print("Train :", train_df.count())
print("Test  :", test_df.count())

In [0]:
## Checking class distribution
print("="*60)
print("CLASS DISTRIBUTION - TRAIN DATA")
print("="*60)

display(
    train_df.groupBy("is_anomaly")
            .count()
            .orderBy("is_anomaly")
)

In [0]:
## Seperate Majority and Minority Class
normal_df = train_df.filter("is_anomaly == 0")
anomaly_df = train_df.filter("is_anomaly == 1")

print(f"Normal Logs  : {normal_df.count():,}")
print(f"Anomaly Logs : {anomaly_df.count():,}")

In [0]:
## Random Undersampling
from pyspark.sql import functions as F

## Desired Ratio
NORMAL_TO_ANOMALY_RATIO = 5

anomaly_count = anomaly_df.count()

desired_normal_count = anomaly_count * NORMAL_TO_ANOMALY_RATIO

fraction = desired_normal_count / normal_df.count()

balanced_normal_df = normal_df.sample(
    withReplacement = False,
    fraction = fraction,
    seed = 42
)

print(f"Sampling Fraction : {fraction:.4f}")

In [0]:
## Create Balanced Training Dataset
balanced_train_df = (
    balanced_normal_df
    .unionByName(anomaly_df)
    .orderBy(F.rand(seed = 42))
)

print("="*60)
print("BALANCED TRAIN DATASET")
print("="*60)

print(f"Rows : {balanced_train_df.count():,}")

In [0]:
## Validate Balanced Dataset
display(
    balanced_train_df.groupBy("is_anomaly")
                     .count()
                     .orderBy("is_anomaly")
)

In [0]:
## Build Feature Pipeline
from pyspark.ml.feature import(
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

categorical_cols = [
    "service",
    "log_file",
    "log_level",
    "http_method",
    "response_category"
]

numeric_cols = [
    "status_code",
    "response_time",
    "event_hour",
    "event_day",
    "event_month",
    "event_weekday",
    "is_weekend",
    "instance_exists"
]

target = "is_anomaly"

In [0]:
## Create String Indexers
indexers = [
    StringIndexer(
        inputCol = col,
        outputCol = f"{col}_index",
        handleInvalid = "keep"
    )
    for col in categorical_cols
]

In [0]:
## One Hot Encoding
encoders = [
    OneHotEncoder(
        inputCol = f"{col}_index",
        outputCol = f"{col}_vec",
    )
    for col in categorical_cols
]


In [0]:
## Vector Assembler
assembler_inputs = (
    [f"{col}_vec" for col in categorical_cols]
    + numeric_cols
)

assembler = VectorAssembler(
    inputCols = assembler_inputs,
    outputCol = "features"
)

In [0]:
## Create Pipeline
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages = indexers + encoders + [assembler]
)

In [0]:
## Fit Pipeline
pipeline_model = pipeline.fit(balanced_train_df)

train_features = pipeline_model.transform(balanced_train_df)

test_features = pipeline_model.transform(test_df)

In [0]:
## Validate Features
display(
    train_features.select(
        "features",
        "is_anomaly"
    ).limit(10)
)

In [0]:
## Train Logistic Regression
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol ="features",
    labelCol = "is_anomaly",
    predictionCol = "prediction",
    rawPredictionCol = "rawPrediction",
    maxIter = 100,
    regParam = 0.01,
    elasticNetParam = 0.0
)

lr_model = lr.fit(train_features)
lr_predictions = lr_model.transform(test_features)

In [0]:
## Evaluate Logistic Regression
lr_metrics = classification_report(lr_predictions)

display(lr_predictions.select(
    "is_anomaly",
    "prediction",
    "probability"
).limit(10))

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS `log-analytics`.gold.mlflow_volume;

In [0]:
## Log to MLflow
with mlflow.start_run(run_name = "Binary_Logistic_Regression"):

    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("regParam", 0.01)

    log_metrics(lr_metrics)

    mlflow.spark.log_model(
        lr_model,
        artifact_path = "model",
        dfs_tmpdir="/Volumes/log-analytics/gold/mlflow_volume"
    )

print("✅ Logistic Regression Logged Successfully")

In [0]:
## Decision Tree
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol = "features",
    labelCol = "is_anomaly",
    predictionCol = "prediction",
    maxDepth = 10,
    seed = 42
)

dt_model = dt.fit(train_features)

dt_predictions = dt_model.transform(test_features)

In [0]:
## Evaluate Decison Tree
dt_metrics = classification_report(dt_predictions)

In [0]:
## Log Decision Tree
with mlflow.start_run(run_name = "Binary_Decision_Tree"):

    mlflow.log_param("algorithm","Decision Tree")
    mlflow.log_param("maxDepth", 10)

    log_metrics(dt_metrics)

    mlflow.spark.log_model(
        dt_model,
        artifact_path = "model",
        dfs_tmpdir="/Volumes/log-analytics/gold/mlflow_volume"
    )

print("✅ Decision Tree Logged Successfully")

In [0]:
## Training random forest 
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol = "features",
    labelCol = "is_anomaly",
    predictionCol = "prediction",
    numTrees = 100,
    maxDepth = 12,
    seed = 42
)

rf_model = rf.fit(train_features)

rf_predictions = rf_model.transform(test_features)

In [0]:
## Evaluate Random Forest 
rf_metrics = classification_report(rf_predictions)

In [0]:
## Log Randpm Forest 
with mlflow.start_run(run_name = "Binary_Random_Forest"):

    mlflow.log_param("algorithm","Random Forest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 12)

    log_metrics(rf_metrics)

    mlflow.spark.log_model(
        rf_model,
        artifact_path = "model",
        dfs_tmpdir="/Volumes/log-analytics/gold/mlflow_volume"
    )

print("✅ Random Forest Logged Successfully")

In [0]:
## Compare Models 
comparison_df = compare_models([
    ("Logistic Regression", lr_metrics),
    ("Decision Tree", dt_metrics),
    ("Random Forest", rf_metrics)
])

display(comparison_df)

In [0]:
save_model(
    model=rf_model,
    model_path="/Volumes/log-analytics/gold/mlflow_volume/Binary_Random_Forest"
)